In [2]:
from jupyter_dash import JupyterDash
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd
import sys
import base64

JupyterDash.infer_jupyter_proxy_config()

sys.path.insert(0, '/home/codio/workspace/code_files')
from CRUD_Python_Module import AnimalShelter

username = "aacuser"
password = "MySecurePassword123"
db = AnimalShelter(username, password)

app = JupyterDash(__name__)

# Logo path
logo_path = '/home/codio/workspace/code_files/Grazioso_Salvare_Logo.png'
try:
    with open(logo_path, 'rb') as f:
        encoded_logo = base64.b64encode(f.read()).decode()
    logo = html.Img(src=f'data:image/png;base64,{encoded_logo}', style={'height': '80px', 'margin-right': '20px'})
except Exception as e:
    print(f"Logo error: {e}")
    logo = html.Div()

# CORRECTED filter queries - use sex_upon_outcome, not sex
filter_queries = {
    'Water Rescue': {
        'breed': {'$in': ['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland']},
        'sex_upon_outcome': {'$eq': 'Intact Female'},
        'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
    },
    'Mountain or Wilderness Rescue': {
        'breed': {'$in': ['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog', 'Siberian Husky', 'Rottweiler']},
        'sex_upon_outcome': {'$eq': 'Intact Male'},
        'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}
    },
    'Disaster or Individual Tracking': {
        'breed': {'$in': ['Doberman Pinscher', 'German Shepherd', 'Golden Retriever', 'Bloodhound', 'Rottweiler']},
        'sex_upon_outcome': {'$eq': 'Intact Male'},
        'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
    },
    'Reset': {}
}

# Test queries
print("Testing queries:")
for filter_name, query in filter_queries.items():
    results = db.read(query)
    print(f"{filter_name}: {len(results)} records")

initial_results = db.read({})
initial_data = pd.DataFrame.from_records(initial_results)
if '_id' in initial_data.columns:
    initial_data.drop(columns=['_id'], inplace=True)

app.layout = html.Div([
    html.Div(
        style={'padding': '20px', 'background-color': '#f8f9fa', 'border-bottom': '3px solid #C41E3A', 'display': 'flex', 'align-items': 'center'},
        children=[
            logo,
            html.Div(children=[
                html.H1('Grazioso Salvare Animal Rescue Dashboard', style={'color': '#C41E3A', 'margin': '0', 'font-size': '28px'}),
                html.P('Created by: Heath Davis', style={'font-size': '12px', 'color': '#666', 'margin': '5px 0 0 0'})
            ])
        ]
    ),
    html.Div(
        style={'padding': '20px'},
        children=[
            html.H3('Filter by Rescue Type'),
            dcc.RadioItems(
                id='filter-type',
                options=[{'label': f'  {x}', 'value': x} for x in filter_queries.keys()],
                value='Reset',
                inline=True,
                style={'marginBottom': '10px'}
            )
        ]
    ),
    html.Hr(),
    html.Div(
        style={'padding': '20px', 'overflowX': 'auto'},
        children=[
            html.H3('Animal Records'),
            dash_table.DataTable(
                id='datatable-id',
                columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in initial_data.columns],
                data=initial_data.to_dict('records'),
                page_current=0,
                page_size=10,
                page_action='custom',
                sort_action='custom',
                sort_by=[],
                filter_action='custom',
                filter_query='',
                selected_rows=[],
                style_cell={'textAlign': 'left', 'padding': '10px', 'fontSize': '13px'},
                style_header={'backgroundColor': '#C41E3A', 'fontWeight': 'bold', 'color': 'white', 'padding': '12px'},
                style_data_conditional=[{'if': {'row_index': 'odd'}, 'backgroundColor': '#f9f9f9'}],
                style_table={'maxHeight': '300px', 'overflowY': 'auto'}
            )
        ]
    ),
    html.Hr(),
    html.Div(
        style={'display': 'flex'},
        children=[
            html.Div(id='graph-id', style={'width': '50%', 'padding': '10px'}),
            html.Div(id='map-id', style={'width': '50%', 'padding': '10px'})
        ]
    )
])

@app.callback(
    Output('datatable-id', 'data'),
    Input('filter-type', 'value')
)
def update_datatable(filter_type):
    query = filter_queries.get(filter_type, {})
    results = db.read(query)
    data = []
    for record in results:
        if '_id' in record:
            del record['_id']
        data.append(record)
    return data

@app.callback(
    Output('graph-id', 'children'),
    Input('datatable-id', 'data')
)
def update_graph(data):
    if not data or len(data) == 0:
        return html.P('No data available')
    
    df_filtered = pd.DataFrame(data)
    if 'breed' not in df_filtered.columns:
        return html.P('Breed column not found')
    
    fig = px.pie(df_filtered, names='breed', title='Breed Distribution')
    fig.update_layout(height=400)
    return dcc.Graph(figure=fig)

@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'data'),
     Input('datatable-id', 'derived_virtual_selected_rows')]
)
def update_map(data, selected_rows):
    map_children = [dl.TileLayer()]
    
    if not data:
        return html.Div([
            html.H3('Animal Location Map'),
            dl.Map(style={'width': '100%', 'height': '400px'}, center=(30.5, -97.5), zoom=11, children=map_children)
        ])
    
    dff = pd.DataFrame(data)
    
    if selected_rows and len(selected_rows) > 0:
        row = selected_rows[0]
        if row < len(dff):
            animal = dff.iloc[row]
            if pd.notna(animal.get('location_lat')) and pd.notna(animal.get('location_long')):
                try:
                    lat = float(animal['location_lat'])
                    lon = float(animal['location_long'])
                    map_children.append(
                        dl.Marker(
                            position=[lat, lon],
                            children=[dl.Popup(f"Animal: {animal.get('name', 'Unknown')}")]
                        )
                    )
                except:
                    pass
    else:
        for idx, row in dff.iterrows():
            if pd.notna(row.get('location_lat')) and pd.notna(row.get('location_long')):
                try:
                    lat = float(row['location_lat'])
                    lon = float(row['location_long'])
                    map_children.append(
                        dl.Marker(
                            position=[lat, lon],
                            children=[dl.Popup(f"Animal: {row.get('name', 'Unknown')}")]
                        )
                    )
                except:
                    pass
    
    return html.Div([
        html.H3('Animal Location Map'),
        dl.Map(style={'width': '100%', 'height': '400px'}, center=(30.5, -97.5), zoom=11, children=map_children)
    ])

app.run_server(debug=False, port=8050)

Testing queries:
Water Rescue: 17 records
Mountain or Wilderness Rescue: 6 records
Disaster or Individual Tracking: 2 records
Reset: 10011 records


 * Running on http://127.0.0.1:8050/ (Press CTRL+C to quit)
127.0.0.1 - - [22/Feb/2026 21:32:08] "GET /_alive_9dedcbbf-4fbf-483b-854b-133c9fff2c39 HTTP/1.1" 200 -


Dash app running on https://gammainch-gopherspeech-3000.codio.io/proxy/8050/


127.0.0.1 - - [22/Feb/2026 21:32:14] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [22/Feb/2026 21:32:15] "GET /_dash-dependencies HTTP/1.1" 200 -
127.0.0.1 - - [22/Feb/2026 21:32:15] "GET /_favicon.ico?v=2.8.1 HTTP/1.1" 200 -
127.0.0.1 - - [22/Feb/2026 21:32:15] "GET /_dash-layout HTTP/1.1" 200 -
127.0.0.1 - - [22/Feb/2026 21:32:15] "GET /_dash-component-suites/dash/dash_table/async-table.js HTTP/1.1" 304 -
127.0.0.1 - - [22/Feb/2026 21:32:15] "GET /_dash-component-suites/dash/dash_table/async-highlight.js HTTP/1.1" 304 -
127.0.0.1 - - [22/Feb/2026 21:32:16] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [22/Feb/2026 21:34:21] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [22/Feb/2026 21:34:21] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [22/Feb/2026 21:35:58] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [22/Feb/2026 21:36:54] "GET /_dash-component-suites/dash/dcc/async-graph.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Feb/2026 21:36:55] "GET 